**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 04: Transformers 라이브러리 & HuggingFace Hub

## 📋 학습 목표

1️⃣ HuggingFace Hub에 연결하고 모델을 탐색하는 방법을 배운다  
2️⃣ AutoModel/AutoTokenizer로 모델을 로딩하고 추론한다  
3️⃣ Pipeline API를 사용한 간편 추론을 실습한다  
4️⃣ BitsAndBytes 4bit 양자화로 메모리를 절약한다  
5️⃣ 모델 크기별 GPU 메모리 사용량을 비교한다  
6️⃣ (심화) safetensors → load_state_dict → forward 로 이어지는 웨이트 로딩 과정을 직접 재현한다  

---

### 🖥️ 실습 환경
- **GPU**: NVIDIA RTX 4060 (8GB VRAM)
- **기본 모델**: Qwen2.5-1.5B-Instruct
- **필수 패키지**: transformers, torch, accelerate, bitsandbytes

## 1️⃣ GPU 환경 점검 및 유틸리티

In [1]:
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print("⚠️ GPU를 사용할 수 없습니다.")

def cleanup_gpu():
    """GPU 메모리를 정리합니다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print_gpu_memory("정리 후")

# GPU 정보 확인
if torch.cuda.is_available():
    print(f"✅ GPU 이름: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"📊 VRAM: {props.total_memory / 1024**3:.1f}GB")
    print(f"📊 CUDA Capability: {props.major}.{props.minor}")
    print_gpu_memory("초기 상태")
else:
    print("⚠️ CUDA GPU를 찾을 수 없습니다.")

✅ GPU 이름: NVIDIA TITAN RTX
📊 VRAM: 23.5GB
📊 CUDA Capability: 7.5
[초기 상태] GPU: 0.0GB / 23.5GB


## 2️⃣ 필수 패키지 확인

Transformers 생태계의 핵심 패키지들을 확인합니다.

In [2]:
# 📦 패키지 버전 확인
import importlib

packages = [
    "transformers",
    "torch",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
]

print("📦 패키지 버전 확인")
print("=" * 40)

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "unknown")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치되지 않음")
        print(f"     pip install {pkg_name}")

📦 패키지 버전 확인


/home/ejkim/multicampus/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ejkim/multicampus/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


  ✅ transformers: 4.57.6
  ✅ torch: 2.10.0+cu128
  ✅ accelerate: 1.13.0
  ✅ bitsandbytes: 0.49.2
  ✅ huggingface_hub: 0.36.2


## 3️⃣ HuggingFace Hub 연결

HuggingFace Hub은 모델, 데이터셋, Space를 공유하는 플랫폼입니다.

### 🔑 Hub 기능
- 🔹 수만 개의 사전학습 모델 제공
- 🔹 모델 카드로 성능/사용법 확인
- 🔹 API 토큰으로 비공개 모델 접근
- 🔹 모델 자동 다운로드 및 캐싱

In [3]:
from huggingface_hub import HfApi, list_models

# 🔍 HuggingFace Hub에서 모델 검색
api = HfApi()

print("🔍 HuggingFace Hub에서 Qwen2.5 모델 검색")
print("=" * 60)

models = list(api.list_models(
    search="Qwen2.5",
    sort="downloads",
    limit=10,
))

for i, model in enumerate(models, 1):
    downloads = model.downloads if hasattr(model, 'downloads') else 'N/A'
    print(f"  {i}. {model.id}")
    print(f"     📥 다운로드: {downloads:,}")
    print()

🔍 HuggingFace Hub에서 Qwen2.5 모델 검색
  1. Qwen/Qwen2.5-7B-Instruct
     📥 다운로드: 17,413,142

  2. Qwen/Qwen2.5-1.5B-Instruct
     📥 다운로드: 9,628,240

  3. Qwen/Qwen2.5-3B-Instruct
     📥 다운로드: 7,618,286

  4. Qwen/Qwen2.5-VL-3B-Instruct
     📥 다운로드: 6,654,181

  5. Qwen/Qwen2.5-0.5B-Instruct
     📥 다운로드: 5,613,212

  6. Qwen/Qwen2.5-VL-7B-Instruct
     📥 다운로드: 4,585,865

  7. Qwen/Qwen2.5-32B-Instruct
     📥 다운로드: 4,260,030

  8. Qwen/Qwen2.5-Coder-7B-Instruct
     📥 다운로드: 2,557,597

  9. Qwen/Qwen2.5-14B-Instruct-AWQ
     📥 다운로드: 1,969,054

  10. Qwen/Qwen2.5-0.5B
     📥 다운로드: 1,834,226



In [4]:
# 🔐 HuggingFace 로그인 (선택사항)
# 비공개 모델이나 게이티드 모델에 접근할 때 필요합니다.

# 방법 1: 환경변수로 설정
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# 방법 2: huggingface-cli login
# !huggingface-cli login

# 방법 3: 코드에서 직접
# from huggingface_hub import login
# login(token="hf_your_token_here")

print("📌 HuggingFace 토큰 설정 방법 안내:")
print("  1. https://huggingface.co/settings/tokens 에서 토큰 발급")
print("  2. 위 코드에서 원하는 방법으로 설정")
print("  3. Qwen2.5 모델은 공개 모델이므로 토큰 없이도 사용 가능")

📌 HuggingFace 토큰 설정 방법 안내:
  1. https://huggingface.co/settings/tokens 에서 토큰 발급
  2. 위 코드에서 원하는 방법으로 설정
  3. Qwen2.5 모델은 공개 모델이므로 토큰 없이도 사용 가능


## 4️⃣ AutoModel / AutoTokenizer로 모델 로딩

Transformers 라이브러리의 `Auto` 클래스를 사용하면, 모델 이름만으로 자동으로 적절한 클래스를 선택합니다.

### 📚 주요 Auto 클래스
- `AutoModelForCausalLM` - 텍스트 생성 (GPT 계열)
- `AutoModelForSeq2SeqLM` - 시퀀스-투-시퀀스 (T5 계열)
- `AutoTokenizer` - 토크나이저 자동 선택

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"📥 모델 로딩 시작: {MODEL_NAME}")
print("⏳ 처음 실행 시 다운로드에 시간이 걸립니다...")
print_gpu_memory("로딩 전")

start_time = time.time()

# 1. 토크나이저 로딩
print("\n🔤 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  ✅ 토크나이저 로딩 완료")
print(f"  📊 어휘 크기: {tokenizer.vocab_size:,}")

# 2. 모델 로딩 (FP16)
print("\n🧠 모델 로딩 중 (FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

elapsed = time.time() - start_time
print(f"  ✅ 모델 로딩 완료! ({elapsed:.1f}초)")
print_gpu_memory("모델 로딩 후")

# 모델 정보 출력
total_params = sum(p.numel() for p in model.parameters())
print(f"\n📊 모델 정보:")
print(f"  파라미터 수: {total_params:,} ({total_params/1e9:.2f}B)")
print(f"  모델 타입: {model.config.model_type}")
print(f"  히든 크기: {model.config.hidden_size}")
print(f"  레이어 수: {model.config.num_hidden_layers}")
print(f"  어텐션 헤드 수: {model.config.num_attention_heads}")

📥 모델 로딩 시작: Qwen/Qwen2.5-1.5B-Instruct
⏳ 처음 실행 시 다운로드에 시간이 걸립니다...
[로딩 전] GPU: 0.0GB / 23.5GB

🔤 토크나이저 로딩 중...
  ✅ 토크나이저 로딩 완료
  📊 어휘 크기: 151,643

🧠 모델 로딩 중 (FP16)...


`torch_dtype` is deprecated! Use `dtype` instead!


  ✅ 모델 로딩 완료! (4.4초)
[모델 로딩 후] GPU: 2.9GB / 23.5GB

📊 모델 정보:
  파라미터 수: 1,543,714,304 (1.54B)
  모델 타입: qwen2
  히든 크기: 1536
  레이어 수: 28
  어텐션 헤드 수: 12


## 5️⃣ 직접 추론하기 (Manual Generation)

토크나이저와 모델을 직접 사용하여 텍스트를 생성합니다.

In [6]:
# 🔤 토크나이저 동작 이해하기
text = "인공지능은 미래를 바꿀 것입니다."

print("🔤 토크나이저 동작 이해")
print("=" * 50)
print(f"원본 텍스트: {text}")

# 인코딩
tokens = tokenizer.encode(text)
print(f"\n📊 토큰 ID: {tokens}")
print(f"📊 토큰 수: {len(tokens)}")

# 개별 토큰 확인
print(f"\n🔍 개별 토큰 분석:")
for i, token_id in enumerate(tokens):
    token_str = tokenizer.decode([token_id])
    print(f"  {i}: ID={token_id:6d} → '{token_str}'")

# 디코딩
decoded = tokenizer.decode(tokens)
print(f"\n🔄 디코딩 결과: {decoded}")

🔤 토크나이저 동작 이해
원본 텍스트: 인공지능은 미래를 바꿀 것입니다.

📊 토큰 ID: [31328, 78125, 21329, 66019, 33704, 143005, 18411, 81718, 144447, 130882, 13]
📊 토큰 수: 11

🔍 개별 토큰 분석:
  0: ID= 31328 → '인'
  1: ID= 78125 → '공'
  2: ID= 21329 → '지'
  3: ID= 66019 → '능'
  4: ID= 33704 → '은'
  5: ID=143005 → ' 미래'
  6: ID= 18411 → '를'
  7: ID= 81718 → ' 바'
  8: ID=144447 → '꿀'
  9: ID=130882 → ' 것입니다'
  10: ID=    13 → '.'

🔄 디코딩 결과: 인공지능은 미래를 바꿀 것입니다.


In [7]:
# 🧠 Chat 템플릿을 사용한 추론
print("🧠 Chat 템플릿을 사용한 추론")
print("=" * 50)

messages = [
    {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요."},
    {"role": "user", "content": "파이썬의 장점 3가지를 알려주세요."}
]

# Chat 템플릿 적용
text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"📝 적용된 템플릿:\n{text_input}")
print(f"\n{'=' * 50}")

# 토큰화 및 추론
inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
print(f"\n📊 입력 토큰 수: {inputs['input_ids'].shape[1]}")

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

elapsed = time.time() - start_time

# 생성된 토큰만 디코딩
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"\n💬 모델 응답:\n{response}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
print(f"📊 생성 토큰 수: {len(generated_ids)}")
print(f"⚡ 처리 속도: {len(generated_ids)/elapsed:.1f} tokens/sec")
print_gpu_memory("추론 후")

🧠 Chat 템플릿을 사용한 추론
📝 적용된 템플릿:
<|im_start|>system
당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요.<|im_end|>
<|im_start|>user
파이썬의 장점 3가지를 알려주세요.<|im_end|>
<|im_start|>assistant



📊 입력 토큰 수: 47

💬 모델 응답:
1) 개발 툴의 다양성과 까다로운 작업을 효율적으로 처리할 수 있는 라이브러리와 프레임워크의 폭广.
2) 강력한 데이터 분석과 전처리 기능.
3) 다양한 언어를 이해하고 번역할 수 있어, 국제적인 협업이 용이합니다.

⏱️ 소요 시간: 4.35초
📊 생성 토큰 수: 82
⚡ 처리 속도: 18.9 tokens/sec
[추론 후] GPU: 2.9GB / 23.5GB


## 6️⃣ Pipeline API를 사용한 간편 추론

`pipeline`은 모델 로딩부터 추론까지 한 줄로 처리하는 고수준 API입니다.

### 📋 주요 Pipeline 타입
- `text-generation` - 텍스트 생성
- `text-classification` - 텍스트 분류
- `question-answering` - 질의응답
- `summarization` - 요약
- `translation` - 번역

In [ ]:
from transformers import pipeline

# 🔧 이미 로딩된 모델로 Pipeline 생성
print("🔧 Pipeline 생성 중...")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("✅ Pipeline 생성 완료!")

# Pipeline으로 추론
print("\n💬 Pipeline 추론 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "간결하게 한국어로 답변하세요."},
    {"role": "user", "content": "딥러닝이란 무엇인가요?"}
]

start_time = time.time()

result = pipe(
    messages,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True, # 만약 do_sample=False로 설정하면 가장 확률이 높은 토큰만 선택하여 더 결정론적인 출력을 생성합니다 (Greedy)
)

elapsed = time.time() - start_time

response_text = result[0]["generated_text"][-1]["content"]
print(f"📝 응답:\n{response_text}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")

Device set to use cuda:0


🔧 Pipeline 생성 중...
✅ Pipeline 생성 완료!

💬 Pipeline 추론 테스트
📝 응답:
딥러닝은 컴퓨터가 데이터를 학습하고 이를 기반으로 새로운 정보를 생성하거나 예측하는 알고리즘입니다. 이는 인공지능의 핵심 기술 중 하나로, 많은 분야에서 활용되고 있습니다. 예를 들어, 이미지認識, 음성인식, 자연어 처리 등에 사용됩니다.

⏱️ 소요 시간: 3.55초


In [9]:
# 📝 다양한 질문 테스트
test_questions = [
    "한국의 4계절 특징을 간단히 설명해주세요.",
    "Python에서 리스트와 튜플의 차이점은?",
    "3 + 5 * 2 의 결과는 무엇인가요?",
]

print("📝 다양한 질문 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'─' * 60}")
    print(f"❓ 질문 {i}: {question}")
    
    messages = [
        {"role": "system", "content": "간결하게 한국어로 답변하세요."},
        {"role": "user", "content": question}
    ]
    
    start_time = time.time()
    result = pipe(messages, max_new_tokens=150, temperature=0.5, do_sample=True)
    elapsed = time.time() - start_time
    
    response_text = result[0]["generated_text"][-1]["content"]
    print(f"💬 응답: {response_text[:200]}")
    print(f"⏱️ {elapsed:.2f}초")

📝 다양한 질문 테스트

────────────────────────────────────────────────────────────
❓ 질문 1: 한국의 4계절 특징을 간단히 설명해주세요.
💬 응답: 한국의 4계절은 다음과 같습니다:

1. 겨울: 12월부터 2월, 기온이 매우 낮고 눈이 많이 내립니다.
2. 봄: 3월부터 5월, 단풍나무 등으로 유명한 계절입니다.
3. 여름: 6월부터 8월, 장마가 많은 계절입니다.
4. 가을: 9월부터 11월, 화사한 풍경과 맑은 날씨를 즐길 수 있습니다.

이러한 계절 변화는 한국의 자연환경과 생활 스타일에 큰 영
⏱️ 6.93초

────────────────────────────────────────────────────────────
❓ 질문 2: Python에서 리스트와 튜플의 차이점은?
💬 응답: Python에서 리스트(LIST)와 튜플(TUPLE)은 모두 데이터를 저장하는 자료구조입니다. 그러나 두 가지는 다음과 같은 차이가 있습니다:

1. 접근성:
   - 리스트: 순서에 따라 접근할 수 있습니다.
   - 튜플: 순서에 따라 접근할 수 있지만, 변경 불가능합니다.

2. 속도:
   - 리스트: 더 빠르지만, 수정이 필요할 때 주의해야 합니다
⏱️ 6.69초

────────────────────────────────────────────────────────────
❓ 질문 3: 3 + 5 * 2 의 결과는 무엇인가요?
💬 응답: 3 + 5 * 2의 결과는 13입니다.
⏱️ 0.73초


In [ ]:
# 🧹 FP16 모델 메모리 해제
print("🧹 FP16 모델 메모리 해제")
print_gpu_memory("해제 전")

del model, pipe
cleanup_gpu()

print("✅ FP16 모델이 메모리에서 해제되었습니다.")

## 7️⃣ BitsAndBytes 4bit 양자화

**양자화(Quantization)**는 모델의 가중치를 낮은 정밀도로 변환하여 메모리를 절약하는 기술입니다.

### 📊 양자화 방식 비교

| 방식 | 정밀도 | 메모리 | 품질 저하 |
|------|--------|--------|----------|
| FP32 | 32bit | 기준 | 없음 |
| FP16 | 16bit | 50% | 거의 없음 |
| INT8 | 8bit | 25% | 약간 |
| **NF4** | **4bit** | **12.5%** | **약간~보통** |

### 🔑 BitsAndBytes NF4 양자화
- NormalFloat4 (NF4): 정규분포 기반 4bit 양자화
- QLoRA 논문에서 제안된 기법
- double quantization으로 추가 메모리 절약

In [ ]:
from transformers import BitsAndBytesConfig

# ⚙️ 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4bit 양자화 활성화
    bnb_4bit_quant_type="nf4",            # NormalFloat4 타입
    bnb_4bit_compute_dtype=torch.bfloat16,  # 연산 시 FP16 사용
    bnb_4bit_use_double_quant=True,       # 이중 양자화 (추가 메모리 절약)
)

print("⚙️ BitsAndBytes 4bit 양자화 설정:")
print(f"  🔹 양자화 타입: NF4 (NormalFloat4)")
print(f"  🔹 연산 정밀도: FP16")
print(f"  🔹 이중 양자화: 활성화")
print(f"\n📊 예상 메모리 절약:")
print(f"  1.5B 모델: ~3GB (FP16) → ~1GB (4bit)")
print(f"  7B 모델: ~14GB (FP16) → ~4GB (4bit)")

In [ ]:
# 📥 4bit 양자화 모델 로딩
print(f"📥 4bit 양자화 모델 로딩: {MODEL_NAME}")
print_gpu_memory("로딩 전")

start_time = time.time()

tokenizer_4bit = AutoTokenizer.from_pretrained(MODEL_NAME)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

elapsed = time.time() - start_time
print(f"\n✅ 4bit 모델 로딩 완료! ({elapsed:.1f}초)")
print_gpu_memory("4bit 로딩 후")

# 모델 정보
total_params = sum(p.numel() for p in model_4bit.parameters())
print(f"\n📊 모델 정보:")
print(f"  파라미터 수: {total_params:,}")
print(f"  양자화 상태: 4bit (NF4)")

In [ ]:
# 🔍 4bit 모델 추론 테스트
print("🔍 4bit 양자화 모델 추론 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "간결하게 한국어로 답변하세요."},
    {"role": "user", "content": "파이썬의 장점 3가지를 알려주세요."}
]

text_input = tokenizer_4bit.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer_4bit(text_input, return_tensors="pt").to(model_4bit.device)

start_time = time.time()

with torch.no_grad():
    outputs = model_4bit.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

elapsed = time.time() - start_time

generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer_4bit.decode(generated_ids, skip_special_tokens=True)

print(f"💬 4bit 모델 응답:\n{response}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
print(f"📊 생성 토큰 수: {len(generated_ids)}")
print(f"⚡ 처리 속도: {len(generated_ids)/elapsed:.1f} tokens/sec")
print_gpu_memory("4bit 추론 후")

## 8️⃣ FP16 vs 4bit 메모리 비교

같은 모델의 FP16과 4bit 버전의 GPU 메모리 사용량을 비교합니다.

In [ ]:
# 📊 FP16 vs 4bit 비교 결과 정리
# (위 실습에서 측정된 값을 기반으로 비교합니다)

print("📊 Qwen2.5-1.5B-Instruct 메모리 비교")
print("=" * 55)
print(f"{'항목':<15} {'FP16':>12} {'4bit (NF4)':>12} {'절약률':>10}")
print("-" * 55)

# 이론적 값 (실습 결과에 따라 다를 수 있음)
comparisons = [
    ("모델 크기", "~3.0 GB", "~1.0 GB", "~67%"),
    ("추론 시 피크", "~3.5 GB", "~1.5 GB", "~57%"),
    ("로딩 시간", "~5초", "~10초", "느림"),
    ("추론 속도", "빠름", "약간 느림", "-"),
    ("품질", "기준", "약간 저하", "-"),
]

for item, fp16, q4, saving in comparisons:
    print(f"{item:<15} {fp16:>12} {q4:>12} {saving:>10}")

print(f"\n💡 핵심 포인트:")
print(f"  🔹 4bit 양자화는 메모리를 60~70% 절약")
print(f"  🔹 품질 저하는 대부분의 작업에서 미미")
print(f"  🔹 RTX 4060 (8GB)에서 7B 모델 사용 가능")
print(f"  🔹 로딩 시간은 양자화로 인해 약간 증가")

In [ ]:
# 🧹 4bit 모델 메모리 해제
print("🧹 4bit 모델 메모리 해제")
del model_4bit, tokenizer_4bit
cleanup_gpu()

## 9️⃣ 생성 파라미터 이해

`model.generate()`에서 사용하는 주요 파라미터를 정리합니다.

In [ ]:
# 📖 생성 파라미터 정리
params = [
    ("max_new_tokens", "생성할 최대 토큰 수", "128~512"),
    ("temperature", "랜덤성 조절 (높을수록 다양)", "0.1~1.0"),
    ("top_p", "누적 확률 기반 샘플링", "0.9~0.95"),
    ("top_k", "상위 k개 토큰에서 샘플링", "20~50"),
    ("repetition_penalty", "반복 억제 (1.0=없음)", "1.0~1.3"),
    ("do_sample", "샘플링 활성화 (False=greedy)", "True/False"),
    ("num_beams", "빔 서치 빔 수", "1~5"),
]

print("📖 텍스트 생성 주요 파라미터")
print("=" * 65)
print(f"{'파라미터':<25} {'설명':<25} {'권장값':>12}")
print("-" * 65)
for name, desc, val in params:
    print(f"{name:<25} {desc:<25} {val:>12}")

print(f"\n💡 팁:")
print(f"  🔹 temperature=0: 항상 같은 결과 (결정적)")
print(f"  🔹 temperature=1: 다양한 결과 (창의적)")
print(f"  🔹 코드 생성: temperature=0.2, 대화: temperature=0.7")

In [ ]:
# 🧹 최종 GPU 메모리 확인
cleanup_gpu()

print("\n📌 오늘의 핵심 정리:")
print("  1️⃣ AutoModelForCausalLM으로 LLM을 쉽게 로딩할 수 있다")
print("  2️⃣ apply_chat_template으로 올바른 프롬프트 포맷을 만든다")
print("  3️⃣ Pipeline API는 추론을 한 줄로 단순화한다")
print("  4️⃣ 4bit 양자화로 GPU 메모리를 60~70% 절약한다")
print("  5️⃣ 생성 파라미터로 출력 품질과 다양성을 조절한다")

## 🔟 심화: from_pretrained 딥다이브 — 웨이트는 어떻게 forward에 연결되는가

> 이 섹션은 **CPU만으로** 실행 가능한 심화 학습입니다.
> 구조가 단순하고 작은 **GPT-2 (124M)** 를 사용해, `from_pretrained()` 한 줄 뒤에서
> 일어나는 일을 전부 손으로 재현해 봅니다.

### 💡 핵심 그림

```
HuggingFace Hub                          로컬 파이썬 객체
┌──────────────────┐                     ┌─────────────────────────────┐
│ config.json      │ ── AutoConfig ────▶ │ 모델 스켈레톤 (랜덤 웨이트)  │
│ model.safetensors│ ── load_file ─────▶ │ state_dict {이름: 텐서}      │
└──────────────────┘                     └──────────┬──────────────────┘
                                                    │ load_state_dict()
                                                    │ (이름 문자열 매칭!)
                                                    ▼
                                         ┌─────────────────────────────┐
                                         │ 웨이트가 채워진 모델          │
                                         │ forward() = 이 텐서들의 행렬곱 │
                                         └─────────────────────────────┘
```

### 10.1 Hub에서 "모델"은 그냥 파일 묶음이다

`AutoModel.from_pretrained("gpt2")` 라고 쓰면 마법처럼 모델이 생기는 것 같지만,
실제로 Hub에서 내려받는 것은 **파일 두 개가 핵심**입니다.

| 파일 | 역할 |
|---|---|
| `config.json` | 모델의 **설계도** — 레이어 수, 히든 차원, 헤드 수 등 |
| `model.safetensors` | 모델의 **웨이트** — `{이름: 텐서}` 딕셔너리를 저장한 파일 |

`hf_hub_download`로 파일을 하나씩 직접 받아서 확인해 봅니다.

In [ ]:
from huggingface_hub import hf_hub_download

# from_pretrained 가 내부에서 하는 다운로드를 직접 수행
config_path  = hf_hub_download("gpt2", "config.json")
weights_path = hf_hub_download("gpt2", "model.safetensors")

print("📄 config :", config_path)
print("📦 weights:", weights_path)

import os
print(f"\n웨이트 파일 크기: {os.path.getsize(weights_path) / 1024**2:.0f} MB")

In [ ]:
# config.json = 모델 설계도 (그냥 JSON 텍스트 파일)
import json

with open(config_path) as f:
    config_dict = json.load(f)

print("🔧 GPT-2 설계도 (config.json 핵심 항목)")
print("=" * 40)
for key in ["model_type", "n_layer", "n_head", "n_embd", "vocab_size", "n_positions"]:
    print(f"  {key:12s}: {config_dict[key]}")

### 10.2 safetensors 열어보기 — 웨이트는 `{이름: 텐서}` 딕셔너리

`model.safetensors`는 신비로운 바이너리가 아니라,
**"이름 문자열 → 텐서" 딕셔너리를 통째로 저장한 파일**입니다.

이름에 규칙이 있습니다: `h.{레이어번호}.{서브모듈}.{weight|bias}`
→ 이 이름이 나중에 파이썬 모델의 **모듈 경로와 정확히 일치**해야 할당이 됩니다.

In [ ]:
from safetensors.torch import load_file

state_dict = load_file(weights_path)   # {이름: 텐서} 딕셔너리로 로드

print(f"📦 텐서 개수: {len(state_dict)}개")
print(f"📦 파라미터 총합: {sum(v.numel() for v in state_dict.values()) / 1e6:.1f}M")
print()
print(f"{'이름':45s} {'shape'}")
print("=" * 65)
for k in sorted(state_dict.keys())[:12]:
    print(f"{k:45s} {tuple(state_dict[k].shape)}")
print("  ... (이하 생략)")

In [ ]:
# 레이어 0 하나만 뽑아보면 Transformer 블록 구조가 그대로 보인다
print("🔍 h.0 (Transformer 블록 0번)의 웨이트들")
print("=" * 65)
for k in sorted(state_dict.keys()):
    if k.startswith("h.0."):
        print(f"  {k:43s} {tuple(state_dict[k].shape)}")

# c_attn: (768, 2304) → Q,K,V 세 개를 한 행렬에 합쳐놓은 것 (768×3=2304)
# c_proj: attention 출력 프로젝션
# mlp.c_fc / mlp.c_proj: FFN의 확장(768→3072) / 축소(3072→768)

### 10.3 설계도만으로 모델 스켈레톤 만들기 (웨이트는 랜덤!)

`AutoModelForCausalLM.from_config(config)`는 **구조만** 만듭니다.
이 시점의 웨이트는 랜덤 초기화 상태 — 체크포인트와 전혀 다릅니다.

핵심 관찰: 모델의 `named_parameters()` 이름과 state_dict의 키가 **같은 이름 체계**를 씁니다.
`nn.Module`은 속성 경로(`transformer.h[0].attn.c_attn.weight`)를
그대로 문자열 이름(`h.0.attn.c_attn.weight`)으로 쓰기 때문입니다.

In [ ]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM

config = AutoConfig.from_pretrained("gpt2")
model_random = AutoModelForCausalLM.from_config(config)   # ⚠️ 웨이트 랜덤!

print("🏗️ 모델 파라미터 이름 (named_parameters) — state_dict 키와 같은 체계")
print("=" * 65)
for i, (name, p) in enumerate(model_random.named_parameters()):
    if i < 8:
        print(f"  {name:55s} {tuple(p.shape)}")
print("  ...")

In [ ]:
# 같은 이름의 텐서를 비교 → 아직 랜덤이므로 당연히 다르다
ckpt_w = state_dict["h.0.attn.c_attn.weight"]              # 체크포인트 파일의 텐서
rand_w = model_random.transformer.h[0].attn.c_attn.weight  # 모델 안의 (랜덤) 텐서

print("체크포인트 텐서 [0,:5]:", ckpt_w[0, :5].tolist())
print("랜덤 모델   텐서 [0,:5]:", rand_w[0, :5].detach().tolist())
print()
print("❓ 두 텐서가 같은가?", torch.equal(ckpt_w, rand_w))  # False

### 10.4 `load_state_dict()` — 이름 매칭으로 웨이트 할당

이제 핵심 한 줄입니다. `load_state_dict(state_dict)`는

1. state_dict의 각 **키(이름)** 를 모델의 파라미터 트리에서 찾고
2. 이름이 일치하면 그 파라미터 텐서에 **값을 복사**합니다

> 매칭 규칙이 **이름 문자열**이기 때문에, 모델 클래스 코드에서 변수명을 바꾸면
> 옛날 체크포인트가 안 열립니다. HF가 모델 코드의 속성명을 함부로 못 바꾸는 이유입니다.

In [ ]:
# GPT2LMHeadModel = transformer(본체) + lm_head 구조이고
# 체크포인트 키는 transformer 기준이므로 model.transformer 에 로드한다
missing, unexpected = model_random.transformer.load_state_dict(state_dict, strict=False)

print("📥 load_state_dict 결과")
print("  missing    (모델에 있는데 파일에 없음):", missing)
print("  unexpected (파일에 있는데 모델에 없음):", unexpected)
print()
print("💡 h.*.attn.bias 는 causal mask 버퍼 — 학습 파라미터가 아니라서")
print("   최신 transformers는 즉석 생성하므로 unexpected 로 떠도 정상입니다.")

In [ ]:
# 로드 후 다시 비교 → 이제 체크포인트 텐서와 완전히 같다!
loaded_w = model_random.transformer.h[0].attn.c_attn.weight

print("체크포인트 텐서 [0,:5]:", ckpt_w[0, :5].tolist())
print("로드된 모델 텐서 [0,:5]:", loaded_w[0, :5].detach().tolist())
print()
print("✅ 두 텐서가 같은가?", torch.equal(ckpt_w, loaded_w))  # True

In [ ]:
# lm_head 는? → GPT-2 는 weight tying: 출력층이 입력 임베딩(wte)을 재사용
model_random.tie_weights()

wte_ptr  = model_random.transformer.wte.weight.data_ptr()
head_ptr = model_random.lm_head.weight.data_ptr()

print("wte.weight     메모리 주소:", hex(wte_ptr))
print("lm_head.weight 메모리 주소:", hex(head_ptr))
print()
print("✅ 같은 메모리를 공유(tied)?", wte_ptr == head_ptr)
print("   → 그래서 state_dict에 lm_head.weight 가 따로 없어도 된다")

### 10.5 검증: `from_pretrained()` = 지금까지 과정의 자동화

`from_pretrained("gpt2")` 한 줄이 내부에서 하는 일:

```
1. Hub에서 config.json / model.safetensors 다운로드   ← 우리가 1️⃣에서 한 것
2. config로 모델 스켈레톤 생성                        ← 3️⃣
3. safetensors 를 state_dict 로 로드                  ← 2️⃣
4. 이름 매칭으로 load_state_dict                      ← 4️⃣
5. tie_weights, eval 모드 등 마무리                   ← 4️⃣
```

두 모델의 forward 출력(logits)이 완전히 같으면 증명 끝입니다.

In [ ]:
from transformers import AutoTokenizer

model_hf = AutoModelForCausalLM.from_pretrained("gpt2")   # 공식 루트
model_hf.eval()
model_random.eval()   # 우리가 수동으로 웨이트를 채운 모델

tokenizer = AutoTokenizer.from_pretrained("gpt2")
input_ids = tokenizer("Hello, my name is", return_tensors="pt").input_ids
print("입력 토큰:", input_ids.tolist())

with torch.no_grad():
    logits_manual = model_random(input_ids).logits   # 수동 로드 모델
    logits_hf     = model_hf(input_ids).logits       # from_pretrained 모델

print()
print("수동 로드 logits[0,-1,:5]      :", logits_manual[0, -1, :5].tolist())
print("from_pretrained logits[0,-1,:5]:", logits_hf[0, -1, :5].tolist())
print()
print("✅ 두 모델의 출력이 동일한가?", torch.allclose(logits_manual, logits_hf, atol=1e-5))

### 10.6 최종 증명: state_dict 텐서만으로 forward 직접 구현

`nn.Module` 없이, **파일에서 읽은 텐서들의 행렬곱만으로** GPT-2 forward를 구현합니다.
이 코드가 HF 모델과 같은 logits를 내면 다음이 증명됩니다:

> **"모델 실행"이란 결국 체크포인트의 텐서들을 정해진 순서(=아키텍처)로 곱하는 것**

📐 GPT-2 forward 한 스텝:
```
x = wte[토큰] + wpe[위치]                          # 임베딩
for l in range(12):                                # 12개 블록
    x = x + Attention(LN1(x))                      #   h.{l}.attn.*  텐서 사용
    x = x + MLP(LN2(x))                            #   h.{l}.mlp.*   텐서 사용
x = LN_f(x)                                        # ln_f.* 텐서
logits = x @ wte.T                                 # weight tying
```

⚠️ GPT-2는 선형층에 `Conv1D`를 써서 웨이트가 `(입력, 출력)` 모양으로 저장되어 있습니다.
→ `nn.Linear`의 `(출력, 입력)`과 반대라서, 전치 없이 `x @ W` 그대로 곱하면 됩니다.

In [ ]:
import math

sd = state_dict            # 파일에서 읽은 {이름: 텐서} — 이것만 사용한다!
n_layer = config.n_layer   # 12
n_head  = config.n_head    # 12
n_embd  = config.n_embd    # 768


def layer_norm(x, w, b, eps=1e-5):
    mu  = x.mean(-1, keepdim=True)
    var = x.var(-1, keepdim=True, unbiased=False)
    return (x - mu) / torch.sqrt(var + eps) * w + b


def gelu(x):
    # GPT-2 가 쓰는 tanh 근사 GELU
    return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)))


def manual_forward(ids):
    """safetensors 에서 읽은 텐서만으로 GPT-2 forward 계산"""
    B, T = ids.shape

    # ── 임베딩: 룩업 테이블에서 행 뽑기 ──
    x = sd["wte.weight"][ids] + sd["wpe.weight"][:T]          # (B, T, 768)

    for l in range(n_layer):
        p = f"h.{l}."

        # ── Multi-Head Attention ──
        h = layer_norm(x, sd[p + "ln_1.weight"], sd[p + "ln_1.bias"])
        qkv = h @ sd[p + "attn.c_attn.weight"] + sd[p + "attn.c_attn.bias"]  # (B,T,2304)
        q, k, v = qkv.split(n_embd, dim=-1)                   # Q,K,V 로 분리
        hd = n_embd // n_head                                 # 헤드당 차원 64
        q = q.view(B, T, n_head, hd).transpose(1, 2)          # (B, 12, T, 64)
        k = k.view(B, T, n_head, hd).transpose(1, 2)
        v = v.view(B, T, n_head, hd).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)       # attention score
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool)) # causal mask
        att = att.masked_fill(~mask, float("-inf")).softmax(-1)
        h = (att @ v).transpose(1, 2).reshape(B, T, n_embd)   # 헤드 합치기
        x = x + (h @ sd[p + "attn.c_proj.weight"] + sd[p + "attn.c_proj.bias"])

        # ── MLP (FFN) ──
        h = layer_norm(x, sd[p + "ln_2.weight"], sd[p + "ln_2.bias"])
        h = gelu(h @ sd[p + "mlp.c_fc.weight"] + sd[p + "mlp.c_fc.bias"])    # 768→3072
        x = x + (h @ sd[p + "mlp.c_proj.weight"] + sd[p + "mlp.c_proj.bias"]) # 3072→768

    x = layer_norm(x, sd["ln_f.weight"], sd["ln_f.bias"])
    return x @ sd["wte.weight"].T                             # weight tying: 임베딩 재사용


with torch.no_grad():
    logits_scratch = manual_forward(input_ids)

print("직접 구현 shape:", tuple(logits_scratch.shape), "  (배치, 시퀀스, vocab)")

In [ ]:
# HF 모델과 비교 — 부동소수점 연산 순서 차이만큼의 미세 오차만 존재
max_diff = (logits_scratch - logits_hf).abs().max().item()

print(f"직접 구현 vs HF forward 최대 오차: {max_diff:.2e}")
print("✅ 사실상 동일한가? (atol=1e-3)", torch.allclose(logits_scratch, logits_hf, atol=1e-3))
print()

# 다음 토큰 예측도 비교
next_scratch = logits_scratch[0, -1].argmax()
next_hf      = logits_hf[0, -1].argmax()
print(f'"Hello, my name is" 다음 토큰 예측')
print(f"  직접 구현: {repr(tokenizer.decode(next_scratch))}")
print(f"  HF 모델  : {repr(tokenizer.decode(next_hf))}")

In [ ]:
# 보너스: 직접 구현한 forward 로 자기회귀 생성까지
ids = tokenizer("Hello, my name is", return_tensors="pt").input_ids

with torch.no_grad():
    for _ in range(8):
        logits = manual_forward(ids)
        next_token = logits[0, -1].argmax().view(1, 1)   # greedy
        ids = torch.cat([ids, next_token], dim=1)

print("🎉 safetensors 텐서 + 행렬곱만으로 생성한 문장:")
print(" ", tokenizer.decode(ids[0]))

### 10.7 정리

| 단계 | 코드 | 실체 |
|---|---|---|
| 다운로드 | `hf_hub_download` | config.json + model.safetensors 파일 받기 |
| 웨이트 읽기 | `load_file()` | `{이름: 텐서}` 딕셔너리 |
| 스켈레톤 | `from_config()` | 랜덤 웨이트의 모듈 트리 (이름 체계 동일) |
| 할당 | `load_state_dict()` | **이름 문자열 매칭**으로 텐서 값 복사 |
| 실행 | `forward()` | 채워진 텐서들의 행렬곱 (직접 구현 가능) |

### 💡 이해했으면 답할 수 있는 질문들

1. `from_pretrained` 시 `size mismatch` 에러 → 원인? (config와 체크포인트 shape 불일치)
2. `missing keys` 경고 → 원인? (모델 구조에 있는 이름이 파일에 없음 — 새 헤드 붙일 때 정상)
3. LoRA 어댑터 파일이 작은 이유? (전체가 아닌 일부 이름의 저랭크 텐서만 저장)
4. 왜 `torch_dtype=torch.float16`으로 로드하면 메모리가 반으로? (같은 딕셔너리를 fp16으로 캐스팅해 담을 뿐)


---

## 🎯 실습 과제

1️⃣ `temperature`를 0.1, 0.5, 1.0으로 바꿔가며 같은 질문에 대한 응답 차이를 비교해보세요  
2️⃣ FP16과 4bit 모델에 같은 질문을 해서 품질 차이를 체감해보세요  
3️⃣ `Qwen/Qwen2.5-3B-Instruct` 모델을 4bit로 로딩해보고 메모리를 확인하세요  

---

## 📚 참고 자료
- [HuggingFace Transformers 문서](https://huggingface.co/docs/transformers)
- [BitsAndBytes 문서](https://github.com/TimDettmers/bitsandbytes)
- [Qwen2.5 모델 카드](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)